<a href="https://colab.research.google.com/github/sripriyamallari/AI-FAQ-Chatbot/blob/main/RAG_Document_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Module 4 - RAG, Embeddings & Vector Databases
# Project: AI Document Chatbot using RAG

print("RAG Document Chatbot")
print("Module 4 - RAG, Embeddings & Vector Databases")

RAG Document Chatbot
Module 4 - RAG, Embeddings & Vector Databases


In [2]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 68.6 MB/s eta 0:00:00


In [4]:
import os
import numpy as np
import faiss
import torch

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

print("Uploaded file:", file_name)

Saving AI_Notes_for_RAG_Project.pdf to AI_Notes_for_RAG_Project.pdf
Uploaded file: AI_Notes_for_RAG_Project.pdf


In [6]:
reader = PdfReader(file_name)

text = ""

for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
        text += page_text + "\n"

print("PDF text extracted successfully!")
print("Number of characters:", len(text))

PDF text extracted successfully!
Number of characters: 4501


In [7]:
print(text[:3000])

Artificial Intelligence – Study Notes
Notes for RAG, Embeddings & Vector Database Project
1. What is Artificial Intelligence?
Artificial Intelligence (AI) is a field of computer science focused on creating systems that can perform tasks
that normally require human intelligence. These tasks include learning, reasoning, understanding
language, recognizing images, solving problems, and making decisions.
2. Machine Learning
Machine Learning (ML) is a branch of AI in which computers learn patterns from data instead of being
explicitly programmed for every task. Common types include supervised learning, unsupervised learning,
and reinforcement learning.
3. Deep Learning
Deep Learning is a type of machine learning that uses neural networks with multiple layers. It is widely
used for image recognition, speech processing, natural language processing, and recommendation
systems.
4. Natural Language Processing
Natural Language Processing (NLP) enables computers to work with human language. Applic

In [8]:
text = text.replace("\n", " ")
text = " ".join(text.split())

print("Cleaned text successfully!")
print(text[:1000])

Cleaned text successfully!
Artificial Intelligence – Study Notes Notes for RAG, Embeddings & Vector Database Project 1. What is Artificial Intelligence? Artificial Intelligence (AI) is a field of computer science focused on creating systems that can perform tasks that normally require human intelligence. These tasks include learning, reasoning, understanding language, recognizing images, solving problems, and making decisions. 2. Machine Learning Machine Learning (ML) is a branch of AI in which computers learn patterns from data instead of being explicitly programmed for every task. Common types include supervised learning, unsupervised learning, and reinforcement learning. 3. Deep Learning Deep Learning is a type of machine learning that uses neural networks with multiple layers. It is widely used for image recognition, speech processing, natural language processing, and recommendation systems. 4. Natural Language Processing Natural Language Processing (NLP) enables computers to work 

In [9]:
chunk_size = 500
overlap = 100

chunks = []

start = 0

while start < len(text):
    end = start + chunk_size
    chunk = text[start:end]

    if chunk.strip():
        chunks.append(chunk)

    start += chunk_size - overlap

print("Total chunks created:", len(chunks))

Total chunks created: 12


In [10]:
for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)


--- Chunk 1 ---
Artificial Intelligence – Study Notes Notes for RAG, Embeddings & Vector Database Project 1. What is Artificial Intelligence? Artificial Intelligence (AI) is a field of computer science focused on creating systems that can perform tasks that normally require human intelligence. These tasks include learning, reasoning, understanding language, recognizing images, solving problems, and making decisions. 2. Machine Learning Machine Learning (ML) is a branch of AI in which computers learn patterns fr

--- Chunk 2 ---
ns. 2. Machine Learning Machine Learning (ML) is a branch of AI in which computers learn patterns from data instead of being explicitly programmed for every task. Common types include supervised learning, unsupervised learning, and reinforcement learning. 3. Deep Learning Deep Learning is a type of machine learning that uses neural networks with multiple layers. It is widely used for image recognition, speech processing, natural language processing, and recomme

In [11]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [12]:
embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embeddings created successfully!")
print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings created successfully!
Embedding shape: (12, 384)


In [13]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings.astype("float32"))

print("FAISS vector database created!")
print("Number of vectors:", index.ntotal)

FAISS vector database created!
Number of vectors: 12


In [14]:
def retrieve_documents(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding.astype("float32"),
        top_k
    )

    results = []

    for i in indices[0]:
        if i < len(chunks):
            results.append(chunks[i])

    return results

In [15]:
query = "What is artificial intelligence?"

results = retrieve_documents(query)

for i, result in enumerate(results):
    print(f"\n--- Retrieved Chunk {i+1} ---")
    print(result)


--- Retrieved Chunk 1 ---
Artificial Intelligence – Study Notes Notes for RAG, Embeddings & Vector Database Project 1. What is Artificial Intelligence? Artificial Intelligence (AI) is a field of computer science focused on creating systems that can perform tasks that normally require human intelligence. These tasks include learning, reasoning, understanding language, recognizing images, solving problems, and making decisions. 2. Machine Learning Machine Learning (ML) is a branch of AI in which computers learn patterns fr

--- Retrieved Chunk 2 ---
ns. 2. Machine Learning Machine Learning (ML) is a branch of AI in which computers learn patterns from data instead of being explicitly programmed for every task. Common types include supervised learning, unsupervised learning, and reinforcement learning. 3. Deep Learning Deep Learning is a type of machine learning that uses neural networks with multiple layers. It is widely used for image recognition, speech processing, natural language pro

In [16]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("LLM loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded successfully!


In [17]:
def generate_answer(question, top_k=3):

    retrieved_chunks = retrieve_documents(
        question,
        top_k
    )

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
Answer the question using only the information provided in the context.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=100
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, retrieved_chunks

In [18]:
question = "What is artificial intelligence?"

answer, sources = generate_answer(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

Question:
What is artificial intelligence?

Answer:
a field of computer science focused on creating systems that can perform tasks that normally require human intelligence


In [19]:
print("Retrieved Context:\n")

for i, source in enumerate(sources):
    print(f"\n--- Source {i+1} ---")
    print(source)

Retrieved Context:


--- Source 1 ---
Artificial Intelligence – Study Notes Notes for RAG, Embeddings & Vector Database Project 1. What is Artificial Intelligence? Artificial Intelligence (AI) is a field of computer science focused on creating systems that can perform tasks that normally require human intelligence. These tasks include learning, reasoning, understanding language, recognizing images, solving problems, and making decisions. 2. Machine Learning Machine Learning (ML) is a branch of AI in which computers learn patterns fr

--- Source 2 ---
ns. 2. Machine Learning Machine Learning (ML) is a branch of AI in which computers learn patterns from data instead of being explicitly programmed for every task. Common types include supervised learning, unsupervised learning, and reinforcement learning. 3. Deep Learning Deep Learning is a type of machine learning that uses neural networks with multiple layers. It is widely used for image recognition, speech processing, natural language p

In [23]:
print("🤖 AI Document Chatbot")
print("Ask questions about your uploaded document.")
print("Type 'exit' to stop.\n")

while True:
    question = input("You: ").strip()

    if question.lower() == "exit":
        print("\nChatbot stopped. 👋")
        break

    if not question:
        print("Please enter a question.\n")
        continue

    try:
        answer, sources = generate_answer(question)

        print("\nAI:", answer)
        print()

    except Exception as e:
        print("\nError:", e)
        print("Please try another question.\n")

🤖 AI Document Chatbot
Ask questions about your uploaded document.
Type 'exit' to stop.

You: What is artificial intelligence 

AI: a field of computer science focused on creating systems that can perform tasks that normally require human intelligence

You: What is RAG

AI: Retrieval-Augmented Generation

You: What is manchine learning 

AI: Machine Learning Machine Learning (ML) is a branch of AI in which computers learn patterns from data instead of being explicitly programmed for every task.

You: Exit

Chatbot stopped. 👋
